<a href="https://colab.research.google.com/github/sahitani/Gen-AI-Project/blob/main/day2_prompt_engineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip install anthropic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 627.7/627.7 kB 5.5 MB/s eta 0:00:00


In [4]:
from google.colab import userdata
import anthropic

ANTHROPIC_API_KEY = userdata.get('ANTHROPIC_API_KEY')
client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)

def ask_claude(prompt, system_prompt="You are a helpful AI assistant.", max_tokens=1024):
    response = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=max_tokens,
        system=system_prompt,
        messages=[{"role": "user", "content": prompt}]
    )
    return response.content[0].text

print("Setup complete ✓")

Setup complete ✓


In [5]:
# Cell 1 — Bad prompt vs Well engineered prompt

question = "order hold issues in supply chain"

# BAD PROMPT — vague, no context, no structure
bad_prompt = "tell me about order hold issues in supply chain"

# WELL ENGINEERED PROMPT — role, task, context, format all defined
good_prompt = """
You are a Supply Chain AI Analyst with 10 years of experience
in Order-to-Cash processes.

I am a Programme Manager working on a process mining engagement
for a large enterprise client experiencing order hold issues.

Analyse the following topic and provide:
1. Top 3 root causes
2. Business impact of each cause
3. Recommended action for each cause

Keep each point concise — maximum 2 lines.

Topic: Order hold issues in supply chain
"""

print("=" * 50)
print("BAD PROMPT OUTPUT:")
print("=" * 50)
print(ask_claude(bad_prompt))

print("\n" + "=" * 50)
print("GOOD PROMPT OUTPUT:")
print("=" * 50)
print(ask_claude(good_prompt))

BAD PROMPT OUTPUT:
# Order Hold Issues in Supply Chain

## What is an Order Hold?

An **order hold** occurs when a sales or purchase order is temporarily suspended from processing, preventing it from moving forward in the fulfillment or procurement cycle. It acts as a **pause mechanism** triggered by specific conditions that need resolution before the order can proceed.

---

## Common Reasons for Order Holds

### 💳 Financial/Credit Holds
- Customer exceeds credit limit
- Overdue invoices or outstanding payments
- Failed payment authorization
- Credit risk assessment flags

### 📦 Inventory/Availability Holds
- Insufficient stock to fulfill the order
- Product discontinuation
- Quality control issues with available inventory
- Awaiting replenishment

### ✅ Compliance & Regulatory Holds
- Export/import restrictions
- Sanctions or embargoes on customers/regions
- Missing documentation (certifications, licenses)
- Customs clearance issues

### 📋 Administrative Holds
- Incomplete or inaccur

In [6]:
# Cell 2 — Few Shot Prompting
# Teaching Claude with examples before asking the real question

few_shot_prompt = """
You are a Supply Chain Analyst classifying order hold issues for Dell.

Here are examples of how we classify order holds:

Example 1:
Issue: Customer purchase order number missing
Classification: HIGH
Reason: Blocks invoice generation and payment cycle completely

Example 2:
Issue: Minor address formatting discrepancy
Classification: LOW
Reason: Can be corrected in next processing cycle without revenue impact

Example 3:
Issue: Credit limit exceeded by 15%
Classification: HIGH
Reason: Requires immediate finance approval before order can proceed

Example 4:
Issue: Product description slightly different from catalogue
Classification: MEDIUM
Reason: Needs verification but does not block immediate processing

Now classify these new order hold issues using exactly the same format:

Issue 1: Tax ID number not matching government records
Issue 2: Shipping label has wrong zip code
Issue 3: Customer requested payment terms not in approved list
"""

print("Few Shot Classification Output:")
print("=" * 50)
print(ask_claude(few_shot_prompt))

Few Shot Classification Output:
Here are the classifications for the three new order hold issues:

---

**Issue 1: Tax ID number not matching government records**
**Classification: HIGH**
**Reason:** Creates serious compliance and legal risk; order cannot proceed until verified with government records, potentially blocking invoicing and shipment entirely

---

**Issue 2: Shipping label has wrong zip code**
**Classification: LOW**
**Reason:** Minor data entry error that can be quickly corrected in the next processing cycle without significant revenue or compliance impact

---

**Issue 3: Customer requested payment terms not in approved list**
**Classification: HIGH**
**Reason:** Requires immediate finance and credit team approval before order can proceed, directly impacting revenue recognition and payment cycle management


In [7]:
# Cell 3 — Chain of Thought Prompting
# Making Claude show its reasoning before giving the answer

# WITHOUT chain of thought
simple_prompt = """
A Dell APJ order has been on hold for 6 days.
Order value is USD 180,000.
Customer is in top 20 accounts.
Credit limit exceeded by 8%.
Should this be escalated to the Finance Director?
"""

# WITH chain of thought
cot_prompt = """
You are a Senior Order Management Analyst at Dell.

A Dell APJ order has been on hold for 6 days.
Order value is USD 180,000.
Customer is in top 20 accounts.
Credit limit exceeded by 8%.

Before giving your final recommendation, think through this
step by step:

Step 1 — Analyse the order value against escalation thresholds
Step 2 — Analyse the hold duration against SLA standards
Step 3 — Analyse the customer importance
Step 4 — Analyse the credit breach severity
Step 5 — Based on all above, give your final recommendation
          with confidence level: High / Medium / Low

Show each step clearly before your final answer.
"""

print("WITHOUT Chain of Thought:")
print("=" * 50)
print(ask_claude(simple_prompt))

print("\n" + "=" * 50)
print("WITH Chain of Thought:")
print("=" * 50)
print(ask_claude(cot_prompt))

WITHOUT Chain of Thought:
# Dell APJ Order Escalation Assessment

## Order Summary
| Parameter | Details |
|-----------|---------|
| Hold Duration | 6 Days |
| Order Value | USD 180,000 |
| Credit Exceedance | 8% |
| Customer Tier | Top 20 Account |

---

## Escalation Recommendation: ✅ YES — Escalate to Finance Director

---

## Key Escalation Triggers

### 🔴 Critical Factors
- **6 days on hold** — Significant delay risking customer satisfaction and relationship damage
- **Top 20 Account** — Strategic customer requiring priority handling and executive attention
- **USD 180,000** — High-value order likely exceeding standard credit team authority limits

### 🟡 Mitigating Factor
- **8% credit exceedance** — Relatively minor breach, suggesting this may be resolvable with a **temporary credit limit adjustment**

---

## Recommended Actions

1. **Immediate escalation** to Finance Director with full context
2. **Loop in Account Executive / Sales Director** given strategic account status
3. *

In [9]:
# Cell 4 Fixed — Handling markdown wrapped JSON
import json

# Clean the response before parsing
def clean_json_response(response):
    # Remove markdown code blocks if present
    cleaned = response.strip()

    if cleaned.startswith("```json"):
        cleaned = cleaned[7:]  # remove ```json from start

    if cleaned.startswith("```"):
        cleaned = cleaned[3:]  # remove ``` from start

    if cleaned.endswith("```"):
        cleaned = cleaned[:-3]  # remove ``` from end

    return cleaned.strip()

# Clean and parse
cleaned_response = clean_json_response(raw_response)
parsed = json.loads(cleaned_response)

# Extract specific fields
analysis = parsed["order_analysis"]

print("Extracted Fields:")
print("=" * 50)
print(f"Risk Level:          {analysis['risk_level']}")
print(f"Recommended Action:  {analysis['recommended_action']}")
print(f"Escalate to Finance: {analysis['escalate_to_finance']}")
print(f"Confidence:          {analysis['confidence_score']}")
print(f"Reasoning:           {analysis['reasoning']}")

print("\n" + "=" * 50)
print("Automated decision based on JSON output:")
print("=" * 50)

if analysis['escalate_to_finance'] == True:
    print("✓ ACTION: Sending escalation alert to Finance Director...")
    print("✓ ACTION: Updating order hold dashboard...")
    print("✓ ACTION: Logging high priority case to database...")
else:
    print("✓ ACTION: Routing to standard processing queue...")

Extracted Fields:
Risk Level:          HIGH
Recommended Action:  Immediately escalate to Finance and Sales leadership to authorize a temporary credit limit increase of at least 10% for this Top 20 account, and expedite order release within 24 hours.
Escalate to Finance: True
Confidence:          HIGH
Reasoning:           An 8% credit breach on a high-value order from a Top 20 account is a low credit risk relative to the customer's strategic importance, and the 6-day hold duration makes immediate escalation and resolution critical to protecting the business relationship.

Automated decision based on JSON output:
✓ ACTION: Sending escalation alert to Finance Director...
✓ ACTION: Updating order hold dashboard...
✓ ACTION: Logging high priority case to database...
